# Base-model CKA — teacher vs student-base representational alignment

**Question.** Is the teacher base (`pythia-160m`) representationally closer to the student
bases it shares **initialization** with (data-seed variants) than to those it shares
**data order** with (weight-seed variants) — and does the answer differ between ordinary
text (Pile) and the number-sequence transfer data?

**Method.** Minibatch linear CKA with the unbiased HSIC₁ estimator
(`scripts.compute_cka` → `pipelines/cka_pipeline.py`), position-sampled rows
(never position 0; numbers condition: digit completion tokens only), all layer pairs
(slot 0 = embeddings, slots 1–12 = raw post-block residuals).

**Reading rules**

- The readout is the **contrast** between pair groups — `primary_data` (ref, dᵢ) vs
  `primary_weight` (ref, wᵢ) — and between data conditions. Absolute CKA values are
  uninterpretable except against the **ceiling** (ref vs itself ⇒ 1.0) and **floor**
  (row-permuted ⇒ ≈ 0) controls. Within-arm pairs are the reference distribution for
  how much CKA varies among "equivalent" models.
- Heatmap orientation is fixed: model A (ref/teacher) layers on **rows**, model B on
  **columns**, input→output from bottom-left; color scale pinned to [0, 1] everywhere.
- Raw residual-stream CKA can saturate under a few dominant directions — a uniformly
  bright matrix means saturation, not similarity; lean on the PC-ablated variants
  (`pc1/pc3/pc5` = top-r PCs projected out per model×layer). A bright ridge *off* the
  diagonal means depth-shifted correspondence (the diagonal then understates similarity).
- Early layers are typically more similar across seeds than later layers — a diagonal
  curve declining with depth is expected, not a bug.
- A flat contrast does **not** falsify representation-mediated transfer: linear CKA
  weights directions by variance and may be blind to a low-variance trait-relevant
  subspace.

**Adding runs.** `uv run python -m scripts.compute_cka --condition pile|numbers ...
--run-tag <tag>` writes to `{output_root}/analysis/cka/<tag>/`; re-run this notebook —
it discovers whatever run tags are on disk (scratch tags listed in `EXCLUDE` are hidden).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from subliminal_learning.analysis.setup import bootstrap, set_plot_style

ROOT, OUTPUT_ROOT = bootstrap()
set_plot_style()
CKA_ROOT = OUTPUT_ROOT / "analysis" / "cka"

GROUP_ORDER = ["primary", "primary_data", "primary_weight", "within_data",
               "within_weight", "ceiling", "self", "floor"]
GROUP_COLORS = {
    "primary": "#2ca02c",         # explicit --model-a/--model-b pair (finetuned mode)
    "primary_data": "#d62728",    # ref vs data-seed (shared init)  — the hypothesis arm
    "primary_weight": "#1f77b4",  # ref vs weight-seed (shared data order)
    "within_data": "#ff9896",
    "within_weight": "#aec7e8",
    "ceiling": "black",
    "self": "#7f7f7f",
    "floor": "#c7c7c7",
}
VARIANT_ORDER = ["raw", "pc1", "pc3", "pc5", "meanpool"]


def load_mat(run_tag, pair, variant):
    return np.load(CKA_ROOT / run_tag / f"cka__{pair}__{variant}.npy")

## 1 · Runs on disk

Run tags are DERIVED, never hand-chosen: `pile`, or `numbers_{generator}_{mode}` read
off the pool path, plus a deterministic `_axis{value}` suffix for any axis deviating
from the canonical defaults (`scripts/compute_cka._CANONICAL`). Canonical runs
(1 position per sequence — every batch row from a different sequence):

- **`pile`** — single-pass auto-sized pool (k × seqs_per_batch = 25,600 docs of
  held-out Pile; no doc revisited).
- **`numbers_base160m_prefix`** — numbers generated by the **teacher base model**
  (untuned `pythia-160m`), `prefix` filter mode; the students' EXACT training set
  (fixed 10k @ seed-42 subsample, the distillation convention). 512 of the 10k rows
  are reserved for PC estimation; the CKA accumulation sees the other 9,488.

Each run's `meta.json` / `cka_manifest.json` records the pool path, file sha256,
tokenized-pool sha256, subsample settings, and skip counts, so provenance is pinned
in artifacts, not tag names.

In [ ]:
EXCLUDE = set()  # scratch run tags to hide from the analysis

# A run is loaded ONLY if its cka_manifest.json exists — the manifest is written
# last by the pipeline, so its presence certifies a complete run produced under
# exactly the recorded identity. Partial/interrupted dirs are skipped loudly.
runs = {}
for run_dir in sorted(p for p in CKA_ROOT.iterdir() if p.is_dir()):
    if run_dir.name in EXCLUDE:
        continue
    if not (run_dir / "cka_manifest.json").exists():
        print(f"SKIPPING {run_dir.name}: no cka_manifest.json (incomplete/interrupted run)")
        continue
    runs[run_dir.name] = json.loads((run_dir / "summary.json").read_text())
assert runs, f"no complete CKA runs under {CKA_ROOT}; run scripts.compute_cka first"

for tag, s in runs.items():
    c = s["meta"]["config"]
    print(f"{tag:32s} condition={c['condition']:8s} k={c['k_batches']:>4} "
          f"m={c['batch_rows']} pos/seq={c['positions_per_seq']} "
          f"models={len(s['meta']['models'])} pool={s['meta']['pool_n_seqs']} seqs")
    print(f"{'':32s} source: {s['meta']['pool_source']}")


In [ ]:
df = pd.DataFrame(
    {"run": tag, "condition": s["meta"]["config"]["condition"], **row}
    for tag, s in runs.items()
    for row in s["rows"]
)
df["group"] = pd.Categorical(df["group"], GROUP_ORDER, ordered=True)
variants = [v for v in VARIANT_ORDER if v in set(df["variant"])]
df["variant"] = pd.Categorical(df["variant"], variants, ordered=True)
RUNS = list(runs)
print(f"{len(df)} pair x variant rows across {len(RUNS)} run(s); variants: {variants}")


## 2 · Summary tables

Per-pair diagonal means first (each cell = mean over the 13 diagonal CKA entries of that
single pair), then the across-pair aggregate per group (mean, min–max over pairs). No
cross-condition averaging anywhere — conditions are separate tables.

In [ ]:
for tag in RUNS:
    d = df[df.run == tag]
    tab = (d.pivot_table(index=["group", "pair"], columns="variant",
                         values="diag_mean", observed=True)
             .reindex(columns=variants).round(3))
    print(f"=== {tag} — per-pair mean diagonal CKA ===")
    display(tab)


In [ ]:
agg = (df.groupby(["run", "variant", "group"], observed=True)["diag_mean"]
         .agg(["mean", "min", "max", "count"]).round(3))
print("Across-pair aggregate per group (mean/min/max over pairs in the group):")
display(agg)


## 3 · Headline contrast — (ref, dᵢ) vs (ref, wᵢ), bracketed by controls

One panel per run. Each dot is one pair's mean diagonal CKA; group means connected
across variants. The question is whether red (shared init) sits above blue (shared
data order), and whether PC ablation opens up a contrast that raw CKA saturates away.

In [ ]:
fig, axes = plt.subplots(1, len(RUNS), figsize=(6.4 * len(RUNS), 4.8),
                         sharey=True, squeeze=False)
x_pos = {v: i for i, v in enumerate(variants)}
rng = np.random.default_rng(0)
for ax, tag in zip(axes[0], RUNS):
    d = df[df.run == tag]
    for grp, dg in d.groupby("group", observed=True):
        xs = dg["variant"].map(x_pos).astype(float) + rng.uniform(-0.09, 0.09, len(dg))
        ax.scatter(xs, dg["diag_mean"], s=90, alpha=0.85, zorder=3,
                   color=GROUP_COLORS[grp], edgecolor="white", linewidth=0.6, label=grp)
        gm = dg.groupby("variant", observed=True)["diag_mean"].mean()
        ax.plot([x_pos[v] for v in gm.index], gm.values, color=GROUP_COLORS[grp],
                linewidth=2.2, alpha=0.65, zorder=2)
    ax.axhline(0, color="0.6", linewidth=0.8)
    ax.axhline(1, color="0.6", linewidth=0.8)
    ax.set_xticks(range(len(variants)), variants)
    ax.set_title(f"{tag}\n(condition: {d['condition'].iloc[0]})")
    ax.set_ylabel("mean diagonal CKA (per pair)")
axes[0, -1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
                   fontsize=11, title="pair group")
fig.tight_layout()
plt.show()


## 4 · Alignment vs depth — per-layer diagonal curves

Diagonal entry (i, i) per layer slot, per pair. Left: raw; right: pc3-ablated.
The floor curve shows the noise level per layer; ceiling pins 1.0.

In [ ]:
show_variants = [v for v in ["raw", "pc3"] if v in variants] or variants[:1]
fig, axes = plt.subplots(len(RUNS), len(show_variants),
                         figsize=(7.0 * len(show_variants), 4.4 * len(RUNS)),
                         sharey=True, squeeze=False)
for r, tag in enumerate(RUNS):
    d = df[df.run == tag]
    for c, variant in enumerate(show_variants):
        ax = axes[r, c]
        for _, row in d[d.variant == variant].iterrows():
            diag = np.diag(load_mat(tag, row["pair"], variant))
            grp = row["group"]
            ax.plot(range(len(diag)), diag, color=GROUP_COLORS[grp], linewidth=2.0,
                    alpha=0.4 if grp in ("self",) else 0.9,
                    marker="o", markersize=4.5, label=grp)
        ax.set_title(f"{tag} — {variant}")
        ax.set_xlabel("layer slot (0 = embeddings, 1-12 = blocks)")
        if c == 0:
            ax.set_ylabel("CKA(i, i)")
        ax.set_ylim(-0.1, 1.05)
handles, labels = axes[0, 0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[0, -1].legend(by_label.values(), by_label.keys(), loc="center left",
                   bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=11,
                   title="pair group")
fig.tight_layout()
plt.show()


## 5 · Cross-layer heatmaps — primary pairs

Full 13×13 CKA matrices, ref layers on rows, variant-model layers on columns,
input→output from bottom-left, scale pinned to [0, 1]. Top row: data-seeds
(shared init); bottom row: weight-seeds (shared data order). A bright ridge on the
diagonal = depth-matched correspondence; uniform brightness = saturation (see the
PC-ablated grid below).

In [ ]:
def primary_grid(tag, variant):
    d = df[(df.run == tag) & (df.variant == variant)]
    rows = [(g, d[d.group == g]["pair"].tolist())
            for g in ("primary", "primary_data", "primary_weight")]
    rows = [(g, ps) for g, ps in rows if ps]
    if not rows:
        print(f"{tag}: no primary-group pairs — skipping grid")
        return
    ncols = max(len(ps) for _, ps in rows)
    fig, axes = plt.subplots(len(rows), ncols,
                             figsize=(3.4 * ncols + 1.2, 3.3 * len(rows)),
                             squeeze=False)
    for r, (grp, pairs_) in enumerate(rows):
        for c in range(ncols):
            ax = axes[r, c]
            if c >= len(pairs_):
                ax.axis("off")
                continue
            mat = load_mat(tag, pairs_[c], variant)
            im = ax.imshow(mat, vmin=0, vmax=1, origin="lower", cmap="magma")
            ax.set_title(f"{pairs_[c]}\ndiag {np.nanmean(np.diag(mat)):.3f}",
                         fontsize=9, color=GROUP_COLORS[grp])
            ax.grid(False)
            if c == 0:
                ax.set_ylabel("model A layer")
            ax.set_xlabel("model B layer")
    fig.colorbar(im, ax=axes, shrink=0.85, label="CKA")
    fig.suptitle(f"{tag} — {variant}", y=1.02)
    plt.show()


for tag in RUNS:
    primary_grid(tag, "raw")

In [ ]:
for tag in RUNS:
    if "pc3" in variants:
        primary_grid(tag, "pc3")


## 6 · The contrast heatmap — mean(ref, dᵢ) − mean(ref, wᵢ)

The layer-pair-resolved version of the headline: positive (red) = data-seed pairs more
aligned with the ref than weight-seed pairs at that layer pair. Diverging scale centered
on 0 (fixed ±0.1; values beyond saturate).

In [ ]:
show_all = [v for v in variants if v != "meanpool"]
# Only PolyPythia-matrix runs have both contrast arms; explicit --model-a/--model-b
# runs (single "primary" pair) have no d-vs-w contrast and are skipped here.
contrast_runs = [
    tag for tag in RUNS
    if not df[(df.run == tag) & (df.group == "primary_data")].empty
    and not df[(df.run == tag) & (df.group == "primary_weight")].empty
]
skipped = [t for t in RUNS if t not in contrast_runs]
if skipped:
    print(f"no d-vs-w arms (explicit-pair runs) — skipping: {skipped}")
if contrast_runs:
    fig, axes = plt.subplots(len(contrast_runs), len(show_all),
                             figsize=(3.6 * len(show_all) + 1.0, 3.4 * len(contrast_runs)),
                             squeeze=False)
    for r, tag in enumerate(contrast_runs):
        d = df[df.run == tag]
        for c, variant in enumerate(show_all):
            ax = axes[r, c]
            # nanmean: pc-ablated matrices can carry NaN cells (near-degenerate
            # self-HSIC; the pipeline warns and lists them in the manifest).
            md_ = np.nanmean([load_mat(tag, p, variant)
                              for p in d[(d.group == "primary_data") & (d.variant == variant)]["pair"]], axis=0)
            mw = np.nanmean([load_mat(tag, p, variant)
                             for p in d[(d.group == "primary_weight") & (d.variant == variant)]["pair"]], axis=0)
            diff = md_ - mw
            im = ax.imshow(diff, vmin=-0.1, vmax=0.1, origin="lower", cmap="RdBu_r")
            ax.set_title(f"{tag} — {variant}\nmean diag Δ {np.nanmean(np.diag(diff)):+.3f}",
                         fontsize=9)
            ax.grid(False)
            if c == 0:
                ax.set_ylabel("ref layer")
            ax.set_xlabel("variant-model layer")
    fig.colorbar(im, ax=axes, shrink=0.85, label="CKA(data) − CKA(weight)")
    plt.show()

## 7 · Within-model self-heatmaps (block-structure diagnostic)

Each model vs itself. A large saturated off-diagonal block means that depth range of the
cross-model heatmaps is uninformative in raw form — lean on the PC-ablated variants
there.

In [ ]:
for tag in RUNS:
    d = df[(df.run == tag) & (df.variant == "raw") & (df.group.isin(["ceiling", "self"]))]
    pairs_ = d["pair"].tolist()
    fig, axes = plt.subplots(1, len(pairs_), figsize=(2.9 * len(pairs_) + 1.0, 3.1),
                             squeeze=False)
    for c, pair in enumerate(pairs_):
        ax = axes[0, c]
        im = ax.imshow(load_mat(tag, pair, "raw"), vmin=0, vmax=1, origin="lower",
                       cmap="magma")
        ax.set_title(pair.removeprefix("self__"), fontsize=9)
        ax.grid(False)
        if c:
            ax.set_yticks([])
    fig.colorbar(im, ax=axes, shrink=0.85, label="CKA")
    fig.suptitle(f"{tag} — within-model layer x layer (raw)", y=1.05)
    plt.show()


## 8 · Convergence — running mean diagonal CKA vs k

Sanity check on the minibatch estimate: curves must be visually flat by the final k.
If a curve is still drifting, extend the run (`--k-batches 200`).

In [ ]:
fig, axes = plt.subplots(1, len(RUNS), figsize=(6.4 * len(RUNS), 4.2),
                         sharey=True, squeeze=False)
for ax, tag in zip(axes[0], RUNS):
    curves = runs[tag]["convergence_curves"]
    for _, row in df[(df.run == tag) & (df.variant == "raw")].iterrows():
        curve = curves[f"{row['pair']}__raw"]
        ax.plot([p["k"] for p in curve], [p["diag_mean"] for p in curve],
                color=GROUP_COLORS[row["group"]], linewidth=1.8, alpha=0.8,
                marker="o", markersize=3.5, label=row["group"])
    ax.set_title(f"{tag} (raw)")
    ax.set_xlabel("k (batches)")
    ax.set_ylabel("running mean diagonal CKA")
handles, labels = axes[0, -1].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axes[0, -1].legend(by_label.values(), by_label.keys(), loc="center left",
                   bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=11,
                   title="pair group")
fig.tight_layout()
plt.show()


## 9 · Methods notes for the writeup

Deliberate deviations from common LLM CKA practice (state these rather than leave
implicit):

1. Most LLM work computes full-batch linear CKA with the **biased** HSIC₀ estimator over
   pooled token positions. We use the Nguyen et al. **minibatch unbiased** estimator with
   independent position sampling (1–4 positions/sequence, never position 0, digit-only
   completion tokens in the numbers condition), which is stricter about the U-statistic's
   independence assumption. The `meanpool` variant (one mean-pooled row per sequence) is
   the faithfulness check against the source paper's one-row-per-example setup — report
   it alongside, don't substitute it for the position-sampled primary.
2. No claim rests on absolute CKA values — only on contrasts bracketed by the
   floor/ceiling controls.
3. A near-identical CKA across arms does NOT falsify representation-mediated transfer
   (variance-weighting blindness to low-variance trait subspaces). A cheap robustness
   add if the contrast carries weight: a complementary geometry metric (e.g. mutual-kNN
   overlap) on the same activation samples.